# Portugal Glaucoma Market — Raw File Automation

End-to-end automation of the IQVIA Santen **Glaucoma** raw file into the dashboard-ready tables (separate from the Portugal AT notebook, same overall workflow).

**Pipeline**
1. **`raw_df`** — reproduce the `Pivot` tab (the "All Data" pivot) from the `All Data` sheet
2. **`extract_df`** — a second pivot in the `Glaucoma Pivot (2)` layout (Geo / Product_final rows, Value block + UNITS block)
3. **`Portugal_Glaucoma_NonConv`** — dashboard tab `Glaucoma_Non Converted`, columns **A–BL**, same names & order
4. **`Portugal_Glaucoma_Conv`** — dashboard tab `Glaucoma_Converted` (unit columns converted by the HU factor)
5. **Export** both tables to Excel

QC cells check record counts and that the unit + revenue **column sums match the Excel** exactly.


In [ ]:
import pandas as pd
import numpy as np
import openpyxl

# ---- File paths (edit if the files live elsewhere) ----
RAW_FILE       = "IQVIA_Santen_Glaucoma_Market__2026_06.xlsx"
MAPPING_FILE   = "Portugal_Mapping.xlsx"                         # the dashboard references its 'Portugal' sheet
DASHBOARD_FILE = "Country_Dashboard_Data_-_Portugal_Linked_Galucoma.xlsx"

RAW_SHEET       = "All Data"
MAPPING_SHEET   = "Portugal"
NONCONV_SHEET   = "Glaucoma_Non Converted"
CONV_SHEET      = "Glaucoma_Converted"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

## Step 1 — Build `raw_df` (the `Pivot` tab)

Reproduces the manual PivotTable on the `Pivot` sheet: rows `Market / Geo / Product / Pack`; measures `Sum of value` then `Sum of Quantity`; columns `Date` (chronological); plus `Product_final = Product & " " & Pack`. Missing month cells stay blank (`NaN`).


In [ ]:
all_data = pd.read_excel(RAW_FILE, sheet_name=RAW_SHEET, header=0)
for c in ["Market", "Geo", "Product", "Pack"]:
    all_data[c] = all_data[c].astype("string").fillna("")

all_data["dt"] = pd.to_datetime(all_data["Date"], format="%b %y")
months       = sorted(all_data["dt"].unique())
month_labels = [pd.Timestamp(m).strftime("%b %y") for m in months]
print(f"{len(months)} months  |  first: {month_labels[0]}  ->  most recent: {month_labels[-1]}")

grouped   = all_data.groupby(["Market", "Geo", "Product", "Pack", "dt"])[["value", "Quantity"]].sum()
val_block = grouped["value"].unstack("dt").reindex(columns=months)
qty_block = grouped["Quantity"].unstack("dt").reindex(columns=months)
val_block.columns = [f"Val {m}" for m in month_labels]
qty_block.columns = [f"Qty {m}" for m in month_labels]

raw_df = pd.concat([val_block, qty_block], axis=1).reset_index()
raw_df["Product_final"] = raw_df["Product"].fillna("") + " " + raw_df["Pack"].fillna("")
print("raw_df shape:", raw_df.shape)
raw_df.head()

## Step 2 — Build `extract_df` (`Glaucoma Pivot (2)` layout)

Rows `Geo`, `Product_final`; a **Value** block (`Sum of <month>`) then a **UNITS** block (`Sum of <month>2`, the trailing `2` being Excel's duplicate-name suffix). Blank cells stay `NaN`.


In [ ]:
val_cols = [f"Val {m}" for m in month_labels]
qty_cols = [f"Qty {m}" for m in month_labels]

grp = raw_df.groupby(["Geo", "Product_final"], sort=True)
ext_val = grp[val_cols].sum(min_count=1)   # keep all-missing cells blank
ext_qty = grp[qty_cols].sum(min_count=1)
extract_df = pd.concat([ext_val, ext_qty], axis=1).reset_index()

EXT_VALUE = [f"Sum of {m}"  for m in month_labels]   # Value block  (dashboard REV source)
EXT_UNITS = [f"Sum of {m}2" for m in month_labels]   # UNITS block  (dashboard UN source)
extract_df.columns = ["Geo", "Product_final"] + EXT_VALUE + EXT_UNITS
print("extract_df shape:", extract_df.shape)
extract_df.head()

## Step 3 — Build `Portugal_Glaucoma_NonConv` (tab `Glaucoma_Non Converted`, cols **A–BL**)

Replicates every column formula, keeping the exact dashboard names & order.

| Col | Name | Logic |
|---|---|---|
| A | Retina market | `"Inculde"` if *Retina* in Category else `"Exclude"` |
| B | Region | `"Portugal"` |
| C | RSM | constant `"-"` |
| D | Territory | mapping `Portugal`: match **Brick** in AB → AD |
| E | Brick | `Geo` from `extract_df` |
| F | ATC4 Class | constant `"-"` |
| G | Category | mapping (key **Product_final** in col C) → col 10 (L) |
| H | Product | `Product_final` from `extract_df` |
| I | Brand | col 2 (D) | 
| J | Company | col 4 (F) |
| K | Pack | col 6 (H) |
| L | Molecule | col 14 (P) |
| M | Sub-Market | col 11 (M) |
| N | Formulation | col 13 (O) |
| O | Ingredient | col 15 (Q) |
| P | Import | col 16 (R) |
| Q–AN | UN- … (units) | `extract_df` **UNITS** block, positionally. Blank → `0`. |
| AO–BL | REV- … (revenue) | `extract_df` **Value** block, positionally. Blank → `0`. |

Note: all the descriptive lookups match on `Product_final` (Product & Pack), not just the product name.


In [ ]:
# ---- mapping lookup blocks (from the 'Portugal' sheet) ----
mp_raw = pd.read_excel(MAPPING_FILE, sheet_name=MAPPING_SHEET, header=None)

def first_seen(keys, values):
    out = {}
    for k, v in zip(keys, values):
        if pd.notna(k):
            k = str(k).strip()
            if k not in out:
                out[k] = v
    return out

# C-block: key = col C (index 2); offset is 1-based within C..R
C_key = mp_raw.iloc[2:, 2]
c_lookup = lambda off: first_seen(C_key, mp_raw.iloc[2:, 2 + off - 1])
category_lookup = c_lookup(10)   # L
brand_lookup    = c_lookup(2)    # D
company_lookup  = c_lookup(4)    # F
pack_lookup     = c_lookup(6)    # H
molecule_lookup = c_lookup(14)   # P
submkt_lookup   = c_lookup(11)   # M
formul_lookup   = c_lookup(13)   # O
ingred_lookup   = c_lookup(15)   # Q
import_lookup   = c_lookup(16)   # R
conv_lookup     = c_lookup(8)    # J  (HU conversion factor -> used only by the Converted table)

# AB-block: key = col AB (27) -> Territory col AD (29)
territory_lookup = first_seen(mp_raw.iloc[2:, 27], mp_raw.iloc[2:, 29])

In [ ]:
def build_glaucoma_table(dashboard_sheet, converted):
    # exact column names / order from the dashboard tab
    wb = openpyxl.load_workbook(DASHBOARD_FILE, read_only=True)
    hdr = next(wb[dashboard_sheet].iter_rows(min_row=1, max_row=1, values_only=True))
    wb.close()
    COLS = list(hdr[:64]); UN = list(hdr[16:40]); REV = list(hdr[40:64])

    def VLOOKUP(series, d):
        return series.map(lambda x: d.get(str(x).strip(), "-") if pd.notna(x) else "-")

    geo = extract_df["Geo"].astype("string")
    pf  = extract_df["Product_final"].astype("string")   # = dashboard 'Product' (H)
    cat = VLOOKUP(pf, category_lookup)

    df = pd.DataFrame({
        COLS[0]:  np.where(cat.astype("string").str.contains("Retina", na=False), "Inculde", "Exclude"),
        COLS[1]:  "Portugal",
        COLS[2]:  "-",
        COLS[3]:  VLOOKUP(geo, territory_lookup),
        COLS[4]:  geo,
        COLS[5]:  "-",
        COLS[6]:  cat,
        COLS[7]:  pf,
        COLS[8]:  VLOOKUP(pf, brand_lookup),
        COLS[9]:  VLOOKUP(pf, company_lookup),
        COLS[10]: VLOOKUP(pf, pack_lookup),
        COLS[11]: VLOOKUP(pf, molecule_lookup),
        COLS[12]: VLOOKUP(pf, submkt_lookup),
        COLS[13]: VLOOKUP(pf, formul_lookup),
        COLS[14]: VLOOKUP(pf, ingred_lookup),
        COLS[15]: VLOOKUP(pf, import_lookup),
    })

    # UN (units) <- extract UNITS block ; REV (revenue) <- extract Value block
    for un_h, uc in zip(UN, EXT_UNITS):
        df[un_h] = pd.to_numeric(extract_df[uc], errors="coerce")

    if converted:
        # =IFERROR(units * HU_factor, units) ; factor from mapping col 8 (J), keyed by Product_final
        cf = pd.to_numeric(pf.map(lambda x: conv_lookup.get(str(x).strip(), None)), errors="coerce")
        for un_h in UN:
            un = pd.to_numeric(df[un_h], errors="coerce")
            df[un_h] = np.where(cf.notna(), un * cf, un)

    for rev_h, vc in zip(REV, EXT_VALUE):
        df[rev_h] = pd.to_numeric(extract_df[vc], errors="coerce")

    df[UN + REV] = df[UN + REV].fillna(0)   # '=<pivot cell>' turns blank into 0
    return df[COLS], UN, REV

Portugal_Glaucoma_NonConv, UN_COLS, REV_COLS = build_glaucoma_table(NONCONV_SHEET, converted=False)
print("Portugal_Glaucoma_NonConv shape:", Portugal_Glaucoma_NonConv.shape)
Portugal_Glaucoma_NonConv.head()

### QC 1 — record count `extract_df` vs `Portugal_Glaucoma_NonConv`

In [ ]:
print("extract_df rows                 :", len(extract_df))
print("Portugal_Glaucoma_NonConv rows  :", len(Portugal_Glaucoma_NonConv))
print("MATCH ✅" if len(extract_df) == len(Portugal_Glaucoma_NonConv) else "MISMATCH ❌")
assert len(extract_df) == len(Portugal_Glaucoma_NonConv)

### QC 2 — sum of every UN and REV column vs the Excel (`Glaucoma_Non Converted`)

In [ ]:
_dash = pd.read_excel(DASHBOARD_FILE, sheet_name=NONCONV_SHEET, header=0).iloc[:, :64]
_rows = []
for c in UN_COLS + REV_COLS:
    s_out = pd.to_numeric(Portugal_Glaucoma_NonConv[c], errors="coerce").sum()
    s_xl  = pd.to_numeric(_dash[c], errors="coerce").sum()
    _rows.append({"Column": c, "Sum_NonConv": s_out, "Sum_Excel": s_xl, "Diff": s_out - s_xl})
qc_sum_nonconv = pd.DataFrame(_rows)
_max = qc_sum_nonconv["Diff"].abs().max()
print(f"Max absolute sum difference: {_max}")
print("ALL COLUMN SUMS MATCH ✅" if _max < 1e-6 else "SUM MISMATCH ❌")
qc_sum_nonconv

## Step 4 — Build `Portugal_Glaucoma_Conv` (tab `Glaucoma_Converted`)

Identical to NonConv, except the **unit** columns Q–AN are converted with the HU factor:

```excel
=IFERROR(<units> * INDEX([1]Portugal!$C$3:$R$2000, MATCH($H2, C-col, 0), 8), <units>)
```

i.e. `units × HU conversion factor` (mapping col 8 = J, keyed by `Product_final`); if the factor isn't found/numeric, the original units are kept. Revenue columns are unchanged.


In [ ]:
Portugal_Glaucoma_Conv, _, _ = build_glaucoma_table(CONV_SHEET, converted=True)
print("Portugal_Glaucoma_Conv shape:", Portugal_Glaucoma_Conv.shape)
Portugal_Glaucoma_Conv.head()

### QC 3 — record count vs `extract_df`

In [ ]:
print("extract_df rows              :", len(extract_df))
print("Portugal_Glaucoma_Conv rows  :", len(Portugal_Glaucoma_Conv))
print("MATCH ✅" if len(extract_df) == len(Portugal_Glaucoma_Conv) else "MISMATCH ❌")
assert len(extract_df) == len(Portugal_Glaucoma_Conv)

### QC 4 — sum of every UN and REV column vs the Excel (`Glaucoma_Converted`)

In [ ]:
_dashc = pd.read_excel(DASHBOARD_FILE, sheet_name=CONV_SHEET, header=0).iloc[:, :64]
_rows = []
for c in UN_COLS + REV_COLS:
    s_out = pd.to_numeric(Portugal_Glaucoma_Conv[c], errors="coerce").sum()
    s_xl  = pd.to_numeric(_dashc[c], errors="coerce").sum()
    _rows.append({"Column": c, "Sum_Conv": s_out, "Sum_Excel": s_xl, "Diff": s_out - s_xl})
qc_sum_conv = pd.DataFrame(_rows)
_max = qc_sum_conv["Diff"].abs().max()
print(f"Max absolute sum difference: {_max}")
print("ALL COLUMN SUMS MATCH ✅" if _max < 1e-6 else "SUM MISMATCH ❌")
qc_sum_conv

## Step 4.5 — Rename month headers to real months, then fill the 42 period columns (BM–DB)

Runs **before** export, on both `Portugal_Glaucoma_NonConv` and `Portugal_Glaucoma_Conv`, without changing anything above.

**A. Header rename** — the template labels on `Q:AN` / `AO:BL` are replaced with the real months, ending at `CURRENT_MONTH` (most recent month in the data): `UN-Jul 2024 … UN-Jun 2026`, `REV-Jul 2024 … REV-Jun 2026`.

**B. 42 period columns (BM–DB)** — 6 blocks (`Current`, `Previous Period` = −1 month, `Previous Year` = −12 months) × 2 measures (`UN`, `REV`) × 7 variants, aggregating each table's own monthly columns. Fiscal year starts in **April**:

- **single** – that month only
- **Fixed Qtr** – from the fiscal-quarter start up to the month (Q1 Apr–Jun, Q2 Jul–Sep, Q3 Oct–Dec, Q4 Jan–Mar)
- **Rolling Qtr** – the month + previous 2 (3 months)
- **Fixed Tercile** – from the tercile start up to the month (T1 Apr–Jul, T2 Aug–Nov, T3 Dec–Mar)
- **Rolling Tercile** – the month + previous 3 (4 months)
- **FY YTD** – from April up to the month (cumulative fiscal year)
- **MAT** – the month + previous 11 (12 months)

The Conv table's UN blocks aggregate its **converted** units; REV blocks match NonConv (revenue isn't converted).


In [ ]:
# ---- CURRENT_MONTH input (leave None to use the most recent month in the data) ----
CURRENT_MONTH = None            # e.g. "Jun 2026"

month_ts   = [pd.Timestamp(m) for m in months]
current_ts = month_ts[-1] if CURRENT_MONTH is None else pd.Timestamp(pd.to_datetime(CURRENT_MONTH, format="%b %Y"))
print("CURRENT_MONTH used:", current_ts.strftime("%b %Y"))

# ---- A. rename Q:AN and AO:BL to the real months (UN-Mon YYYY / REV-Mon YYYY) ----
real_un  = [f"UN-{t.strftime('%b %Y')}"  for t in month_ts]
real_rev = [f"REV-{t.strftime('%b %Y')}" for t in month_ts]
rename_map = {**dict(zip(UN_COLS, real_un)), **dict(zip(REV_COLS, real_rev))}
Portugal_Glaucoma_NonConv = Portugal_Glaucoma_NonConv.rename(columns=rename_map)
Portugal_Glaucoma_Conv    = Portugal_Glaucoma_Conv.rename(columns=rename_map)

# ---- B. period-aggregation helpers (fiscal year starts in April) ----
def _win_size(A, variant):
    fm = (A.month - 4) % 12 + 1                     # fiscal month: Apr=1 ... Mar=12
    return {"single": 1,
            "Fixed Qtr":       ((fm - 1) % 3) + 1,
            "Rolling Qtr":     3,
            "Fixed Tercile":   ((fm - 1) % 4) + 1,
            "Rolling Tercile": 4,
            "FY YTD":          fm,
            "MAT":             12}[variant]

def _period_sum(df, prefix, A, w):
    cols = [f"{prefix}-{(A - pd.DateOffset(months=k)).strftime('%b %Y')}" for k in range(w)]
    cols = [c for c in cols if c in df.columns]     # only months that exist
    if not cols:
        return pd.Series(0.0, index=df.index)
    return df[cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)

_VARIANTS = ["single", "Fixed Qtr", "Rolling Qtr", "Fixed Tercile", "Rolling Tercile", "FY YTD", "MAT"]
_BLOCKS = [("Current",         current_ts),
           ("Previous Period", current_ts - pd.DateOffset(months=1)),
           ("Previous Year",   current_ts - pd.DateOffset(months=12))]

def add_period_columns(df):
    for measure in ["UN", "REV"]:                   # UN blocks first, then REV  -> BM..DB order
        for block, A in _BLOCKS:
            astr = A.strftime("%b %Y")
            for v in _VARIANTS:
                label = f"{block} {measure}-{astr}" if v == "single" else f"{block} {measure}-{v} {astr}"
                df[label] = _period_sum(df, measure, A, _win_size(A, v))
    return df

Portugal_Glaucoma_NonConv = add_period_columns(Portugal_Glaucoma_NonConv)
Portugal_Glaucoma_Conv    = add_period_columns(Portugal_Glaucoma_Conv)

print("Portugal_Glaucoma_NonConv shape:", Portugal_Glaucoma_NonConv.shape)   # -> 106 columns
print("Portugal_Glaucoma_Conv shape:   ", Portugal_Glaucoma_Conv.shape)
Portugal_Glaucoma_NonConv.head()

## Step 5 — Export the processed output files

Writes both tables to Excel in the same processed format (columns **A–BL**, same names/order).


In [ ]:
import os
OUTPUT_DIR = "."   # change to your desired output folder
os.makedirs(OUTPUT_DIR, exist_ok=True)

nonconv_path = os.path.join(OUTPUT_DIR, "Portugal_Glaucoma_NonConv.xlsx")
conv_path    = os.path.join(OUTPUT_DIR, "Portugal_Glaucoma_Conv.xlsx")

Portugal_Glaucoma_NonConv.to_excel(nonconv_path, sheet_name="Glaucoma_Non Converted", index=False)
Portugal_Glaucoma_Conv.to_excel(conv_path,       sheet_name="Glaucoma_Converted",     index=False)

print("Saved:")
print(" ", nonconv_path, Portugal_Glaucoma_NonConv.shape)
print(" ", conv_path,    Portugal_Glaucoma_Conv.shape)

## Recap

- **`raw_df`** / **`extract_df`** — the two Glaucoma pivots
- **`Portugal_Glaucoma_NonConv`** — dashboard `Glaucoma_Non Converted`, cols A–BL; count ✅ and UN+REV sums ✅ vs Excel
- **`Portugal_Glaucoma_Conv`** — dashboard `Glaucoma_Converted` (units × HU factor); count ✅ and sums ✅ vs Excel

Both tables are built from the same `extract_df`, so they are row-aligned with each other. *(pandas row order can differ slightly from Excel's pivot collation; this doesn't affect the count/sum QCs.)*
